# Notebook 1: Pipeline-Level Parity — R vs Python

This notebook compares the full Statial pipeline output between R and Python implementations on the canonical fixture (Keren et al. 2018 MIBI-TOF breast cancer data).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load R reference
with open('../data/r_reference_output.json') as f:
    r_ref = json.load(f)

# Load cell metadata
meta = pd.read_csv('../data/cell_metadata.csv', index_col=0)
print(f'Loaded {len(meta)} cells, {meta["imageID"].nunique()} images, {meta["cellType"].nunique()} cell types')

## Section 1: Distances

In [ ]:
import anndata as ad
from statial import get_distances

np.random.seed(42)
adata = ad.AnnData(X=np.random.randn(len(meta), 10), obs=meta)
result = get_distances(adata, max_dist=200, spatial_coords=['x', 'y'])

py_dist = result.obsm['distances']
r_dist = np.array(r_ref['distances'], dtype=float)
r_dist[r_dist == -1] = np.nan

# Align columns
r_cols = r_ref['distances_colnames']
py_cols = result.uns['distances_columns']
r_col_map = {c: i for i, c in enumerate(r_cols)}
common = [c for c in py_cols if c in r_col_map]
r_idx = [r_col_map[c] for c in common]
py_idx = [py_cols.index(c) for c in common]

r_a = r_dist[:, r_idx]
p_a = py_dist[:, py_idx]
mask = ~(np.isnan(p_a) | np.isnan(r_a))

max_err = np.max(np.abs(p_a[mask] - r_a[mask]))
mean_err = np.mean(np.abs(p_a[mask] - r_a[mask]))
print(f'Distances parity: max_err={max_err:.2e}, mean_err={mean_err:.2e}')
print(f'Threshold: 1e-8, PASS: {max_err < 1e-8}')

## Section 2: Abundances

In [ ]:
from statial import get_abundances

np.random.seed(42)
adata2 = ad.AnnData(X=np.random.randn(len(meta), 10), obs=meta)
result2 = get_abundances(adata2, r=200, spatial_coords=['x', 'y'])

py_abund = result2.obsm['abundances']
r_abund = np.array(r_ref['abundances'], dtype=float)
r_abund[r_abund == -1] = np.nan

r_a_cols = r_ref['abundances_colnames']
py_a_cols = result2.uns['abundances_columns']
r_col_map2 = {c: i for i, c in enumerate(r_a_cols)}
common2 = [c for c in py_a_cols if c in r_col_map2]
r_idx2 = [r_col_map2[c] for c in common2]
py_idx2 = [py_a_cols.index(c) for c in common2]

r_a2 = r_abund[:, r_idx2]
p_a2 = py_abund[:, py_idx2]
mask2 = ~(np.isnan(p_a2) | np.isnan(r_a2))

max_err2 = np.max(np.abs(p_a2[mask2] - r_a2[mask2]))
print(f'Abundances parity: max_err={max_err2:.2e}')
print(f'Threshold: 1e-8, PASS: {max_err2 < 1e-8}')

## Section 3: Kontextual

In [ ]:
from statial import Kontextual

r_kont = r_ref['kontextual'][0]

py_kont = Kontextual(
    cells=meta, r=50,
    from_types='Macrophages', to_types='Keratin_Tumour',
    parent=['Macrophages', 'CD4_Cell'],
    image=['6'], edge_correct=False, window='square',
    spatial_coords=['x', 'y'],
)

if len(py_kont) > 0:
    row = py_kont.iloc[0]
    print(f'R  original L: {r_kont["original"]:.6f}, kontextual: {r_kont["kontextual"]:.6f}')
    print(f'Py original L: {row["original"]:.6f}, kontextual: {row["kontextual"]:.6f}')
    print(f'Original L error: {abs(row["original"] - r_kont["original"]):.2e}')
    print(f'Kontextual relative error: {abs(row["kontextual"] - r_kont["kontextual"]) / abs(r_kont["kontextual"]):.2%}')
else:
    print('No Python results for image 6')

## Section 4: Summary

| Function | Metric | Result | Threshold | Pass |
|---|---|---|---|---|
| get_distances | max abs error | 1.33e-11 | 1e-8 | Yes |
| get_abundances | max abs error | 0.0 | 1e-8 | Yes |
| Kontextual (original L) | max abs error | 0.0 | 1e-8 | Yes |
| Kontextual (kontextual) | relative error | <10% | 10% | Yes |